<a href="https://colab.research.google.com/github/LicaRedMoth/Realtime_Voice_Changer_on_Colab_FIXED/blob/main/Realtime_Voice_Changer_on_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### [w-okada's Voice Changer](https://github.com/w-okada/voice-changer) | **Colab**

---

## **⬇ VERY IMPORTANT ⬇**

You can use the following settings for better results:

If you're using a index: `f0: RMVPE_ONNX | Chunk: 112 or higher | Extra: 8192`<br>
If you're not using a index: `f0: RMVPE_ONNX | Chunk: 96 or higher | Extra: 16384`<br>
**Don't forget to select a T4 GPU in the GPU field, <b>NEVER</b> use CPU!
> Seems that PTH models performance better than ONNX for now, you can still try ONNX models and see if it satisfies you


*You can always [click here](https://github.com/YunaOneeChan/Voice-Changer-Settings) to check if these settings are up-to-date*

---

### <font color=red>⬇ Always use Colab GPU! (**IMPORTANT!**) ⬇</font>
You need to use a Colab GPU so the Voice Changer can work faster and better\
Use the menu above and click on **Runtime** » **Change runtime** » **Hardware acceleration** to select a GPU (**T4 is the free one**)

---
**Credits**<br>
Realtime Voice Changer by [w-okada](https://github.com/w-okada)<br>
Notebook files updated by [rafacasari](https://github.com/Rafacasari)<br>
Recommended settings by [YunaOneeChan](https://github.com/YunaOneeChan)

**Need help?** [AI Hub Discord](https://discord.gg/aihub) » ***#help-realtime-vc***

---

In [ ]:
# @title Clone repository and install dependencies
# @markdown This first step will download the latest version of Voice Changer and install the dependencies. **It can take some time to complete.**
%cd /content/

!pip install colorama --quiet
from colorama import Fore, Style
import os

print(f"{Fore.CYAN}> Cloning the repository...{Style.RESET_ALL}")
!git clone https://github.com/w-okada/voice-changer.git --quiet
print(f"{Fore.GREEN}> Successfully cloned the repository!{Style.RESET_ALL}")
%cd voice-changer/server/

print(f"{Fore.CYAN}> Installing libportaudio2...{Style.RESET_ALL}")
!apt-get -y install libportaudio2 -qq

print(f"{Fore.CYAN}> Installing pre-dependencies...{Style.RESET_ALL}")
# Install dependencies that are missing from requirements.txt and pyngrok
# Downgrade setuptools to fix fairseq build errors
!pip install setuptools==69.5.1 --quiet
!pip install faiss-cpu fairseq pyngrok onnxruntime-gpu librosa sounddevice --quiet
!pip install pyworld --no-build-isolation --quiet
print(f"{Fore.CYAN}> Installing dependencies from requirements.txt...{Style.RESET_ALL}")
!pip install -r requirements.txt --quiet

print(f"{Fore.GREEN}> Successfully installed all packages!{Style.RESET_ALL}")

In [ ]:
# @title Start Server **using ngrok**
# @markdown This cell will start the server, the first time that you run it will download the models, so it can take a while (~1-2 minutes)

# @markdown ---
# @markdown You'll need a ngrok account, but <font color=green>**it's free**</font> and easy to create!
# @markdown ---
# @markdown **1** - Create a <font color=green>**free**</font> account at [ngrok](https://dashboard.ngrok.com/signup) or **login with Google/Github account**\
# @markdown **2** - If you didn't logged in with Google/Github, you will need to **verify your e-mail**!\
# @markdown **3** - Click [this link](https://dashboard.ngrok.com/get-started/your-authtoken) to get your auth token, and place it here:
Token = '' # @param {type:"string"}
# @markdown **4** - *(optional)* Change to a region near to you or keep at United States if increase latency\
# @markdown `Default Region: us - United States (Ohio)`
Region = "eu - Europe (Frankfurt)" # @param ["ap - Asia/Pacific (Singapore)", "au - Australia (Sydney)","eu - Europe (Frankfurt)", "in - India (Mumbai)","jp - Japan (Tokyo)","sa - South America (Sao Paulo)", "us - United States (Ohio)"]

#@markdown **5** - *(optional)* Other options:
ClearConsole = True  # @param {type:"boolean"}

# ---------------------------------
# DO NOT TOUCH ANYTHING DOWN BELOW!
# ---------------------------------

!pip install pyngrok onnxruntime-gpu sounddevice resampy python-socketio --quiet
!pip install fairseq

%cd /content/voice-changer/server

from pyngrok import conf, ngrok
MyConfig = conf.PyngrokConfig()
MyConfig.auth_token = Token
MyConfig.region = Region[0:2]
#conf.get_default().authtoken = Token
#conf.get_default().region = Region
conf.set_default(MyConfig);

import subprocess, threading, time, socket, urllib.request
PORT = 8000

from pyngrok import ngrok
ngrokConnection = ngrok.connect(PORT)
public_url = ngrokConnection.public_url

from IPython.display import clear_output

def wait_for_server():
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', PORT))
        if result == 0:
            break
        sock.close()
    if ClearConsole:
        clear_output()
    print("--------- SERVER READY! ---------")
    print("Your server is available at:")
    print(public_url)
    print("---------------------------------")

threading.Thread(target=wait_for_server, daemon=True).start()

!python3 MMVCServerSIO.py \
  -p {PORT} \
  --https False \
  --content_vec_500 pretrain/checkpoint_best_legacy_500.pt \
  --content_vec_500_onnx pretrain/content_vec_500.onnx \
  --content_vec_500_onnx_on true \
  --hubert_base pretrain/hubert_base.pt \
  --hubert_base_jp pretrain/rinna_hubert_base_jp.pt \
  --hubert_soft pretrain/hubert/hubert-soft-0d54a1f4.pt \
  --nsf_hifigan pretrain/nsf_hifigan/model \
  --crepe_onnx_full pretrain/crepe_onnx_full.onnx \
  --crepe_onnx_tiny pretrain/crepe_onnx_tiny.onnx \
  --rmvpe pretrain/rmvpe.pt \
  --model_dir model_dir \
  --samples samples.json

ngrok.disconnect(ngrokConnection.public_url)

In [ ]:
import requests
import json

# Опрашиваем локальный сервер w-okada
try:
    print("Опрашиваем сервер на 127.0.0.1:8000/api/info...")
    response = requests.get('http://127.0.0.1:8000/api/info')
    if response.status_code == 200:
        info = response.json()
        print("\n=== ТЕКУЩЕЕ СОСТОЯНИЕ СЕРВЕРА ===")

        # Выводим инфу о загруженной модели (0 слот)
        models = info.get('modelSlots', [])
        if len(models) > 0:
            slot_0 = models[0]
            print(f"Слот 0:\n  Имя: {slot_0.get('name')}\n  Загружено: {slot_0.get('isLoaded', False)}\n  Тип: {slot_0.get('voiceChangerType')}")
        else:
            print("Модели не найдены в слотах!")

        # Выводим текущие настройки сервера
        status = info.get('status', {})
        print("\nНастройки:")
        print(f"  Текущий слот: {info.get('serverSettings', {}).get('slot')}")
        print(f"  F0 Detector: {info.get('serverSettings', {}).get('f0Detector')}")
        print(f"  Ожидаемый Chunk Size: {info.get('serverSettings', {}).get('chunkSize')}")
        print(f"  Extra Data Length: {info.get('serverSettings', {}).get('extraConvertSize')}")

        print("\nПолный ответ (первые 500 символов):")
        print(json.dumps(info, indent=2, ensure_ascii=False)[:500] + "...")
    else:
        print(f"Сервер вернул ошибку: {response.status_code}")
except Exception as e:
    print(f"Не удалось подключиться к серверу. Убедитесь, что ячейка с сервером работает! Ошибка: {e}")


### 🛠 Решение: Кастомный локальный Python-клиент

Бэкенд `MMVCServerSIO.py`, который запускается в Colab, уже использует **Socket.IO** для стриминга аудио в реальном времени. Чтобы обойти сломанный UI, вы можете написать скрипт на своем ПК, который будет общаться с сервером Colab через выданный **ngrok URL**.

#### Архитектура:
1. **Colab (Бэкенд)**: Продолжает работать как есть. Вы запускаете ячейку с ngrok, она выдает вам публичный URL (например, `https://xyz.ngrok-free.dev`).
2. **Локальный ПК (Клиент)**:
   - Использует библиотеку `sounddevice` для чтения микрофона и воспроизведения звука.
   - Использует библиотеку `python-socketio` для установки двустороннего соединения с сервером.
   - Отправляет чанки аудио в формате, который ожидает w-okada, и слушает ответные чанки.

#### Пример локального кода (для запуска на вашем ноутбуке)

Для начала установите зависимости на вашем компьютере:
```bash
pip install sounddevice numpy python-socketio[client]
```

Затем создайте файл `vc_client.py`:

```python
import sounddevice as sd
import numpy as np
import socketio
import time

# Вставьте сюда ваш URL от ngrok из Colab (без / на конце)
NGROK_URL = "https://unvictualed-stacia-demonstratedly.ngrok-free.dev"

# Настройки аудио (должны совпадать с сервером)
SAMPLE_RATE = 48000
CHUNK_SIZE = 1024 * 4  # Размер чанка

# Инициализация Socket.IO клиента
sio = socketio.Client()

@sio.event
def connect():
    print("✅ Подключено к Colab серверу!")

@sio.event
def disconnect():
    print("❌ Отключено от сервера.")

# Обработка входящего аудио от сервера
@sio.on('response') # Название эвента зависит от API w-okada (нужно уточнить в исходниках)
def on_audio_response(data):
    # Допустим, сервер возвращает байты обработанного аудио
    audio_chunk = np.frombuffer(data, dtype=np.float32)
    # Воспроизводим звук (в реальном приложении лучше использовать буфер/очередь)
    sd.play(audio_chunk, SAMPLE_RATE)
    sd.wait()

def audio_callback(indata, frames, time_info, status):
    if status:
        print(status)
    # Отправляем записанный чанк на сервер
    sio.emit('request', indata.tobytes())

def main():
    print(f"Подключение к {NGROK_URL}...")
    sio.connect(NGROK_URL)

    print("Начинаем запись и стриминг... Нажмите Ctrl+C для остановки.")
    # Запускаем стриминг с микрофона
    with sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype='float32', callback=audio_callback):
        while True:
            time.sleep(0.1)

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        sio.disconnect()
        print("\nОстановлено.")
```

*Примечание:* События `request` и `response` (имена эвентов Socket.IO) и точный формат байтов (PCM 16-bit или Float32) нужно будет слегка адаптировать под API `MMVCServerSIO.py`. Как альтернатива — вы можете написать в Colab свою минимальную обертку на FastAPI + WebSocket, которая будет импортировать пайплайн w-okada локально и принимать аудио в простом сыром виде.